# Spiral Descent: 条件を変えながら挙動を確認する

この Notebook は、初期状態・風・明示的なモデル仮定をセルごとに確認しながら、Spiral Descent の Reference Path と Trajectory を別々に生成するチュートリアルです。

> **モデル上の注意**: 現在の POH データには、この Spiral Descent 領域を定義する降下性能表がありません。計算結果は `assumed` と明示された局所モデルによるデモで、POH 検証済みの SR22 降下性能予測ではありません。
>
> 学生訓練実施要領の本文から得た Target / Limit / Control Relationship と、章末 Reference Data は分離しています。Reference Data の値を機体性能や固定操作量として使用しません。

## Goal

1. 課目本文から転記した `ManeuverSpec` を確認する。
2. 初期状態・風・モデル仮定を入力する。
3. Reference Path と風の影響を受ける Trajectory を別々に計算する。
4. 結果をグラフで確認し、CSV / KML / PNG を `artifacts/` に保存する。

## Setup

Docker の `notebook` サービスには必要な package が入っています。ローカル実行では repository root で `python3 -m pip install -e '.[notebook]'` を実行してください。

In [ ]:
from pathlib import Path
import os

from IPython.display import display
from matplotlib import pyplot as plt

from sr22_course_simulator.aircraft import (
    FlapSetting,
    GeoPosition,
    InitialState,
    Loading,
    MassItem,
)
from sr22_course_simulator.environment import (
    Atmosphere,
    ConstantWind,
    Environment,
    FlatTerrain,
    NoWind,
)
from sr22_course_simulator.examples.spiral_descent import build_assumption_model
from sr22_course_simulator.export import (
    reference_path_to_kml,
    trajectory_to_kml,
    write_kml,
    write_trajectory_csv,
)
from sr22_course_simulator.geometry import displace_position
from sr22_course_simulator.guidance import (
    SpiralGuidanceConfig,
    simulate_guided_spiral_descent,
)
from sr22_course_simulator.maneuver import spiral_descent_package
from sr22_course_simulator.path import PylonSpiralPath
from sr22_course_simulator.plotting import (
    plot_altitude_time,
    plot_ground_track,
    plot_trajectory_3d,
)
from sr22_course_simulator.provenance import EvidenceKind
from sr22_course_simulator.simulation import (
    AccumulatedTurn,
    SimulationConfig,
    coordinated_turn_radius_m,
)
from sr22_course_simulator.units import (
    degrees_to_radians,
    feet_to_metres,
    knots_to_metres_per_second,
    metres_per_second_to_feet_per_minute,
    metres_per_second_to_knots,
    metres_to_feet,
)

In [ ]:
# Docker では /output、ローカルでは repository の artifacts/ に保存します。
default_artifact_dir = (
    Path.cwd().parent / "artifacts"
    if Path.cwd().name == "notebooks"
    else Path.cwd() / "artifacts"
)
artifact_dir = Path(os.environ.get("SR22_ARTIFACT_DIR", default_artifact_dir)).resolve()
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f"出力先: {artifact_dir}")

## Steps

### 1. 課目本文から得た意味を確認する

ここは入力欄ではありません。Target、Limit、Nominal、Initial Setting、Control Relationship を分けたまま確認します。

In [ ]:
maneuver_package = spiral_descent_package()
maneuver_spec = maneuver_package.spec

print(f"Maneuver: {maneuver_spec.name}")
print(f"Source: {maneuver_spec.source.document_title} / {maneuver_spec.source.section}")
for phase in maneuver_spec.phases:
    print(f"\n[{phase.name}]")
    for role, items in (
        ("target", phase.targets),
        ("limit", phase.limits),
        ("nominal", phase.nominals),
        ("initial_setting", phase.initial_settings),
    ):
        for item in items:
            print(f"  {role}: {item.quantity} = {item.value} {item.unit}")
    for relationship in phase.control_relationships:
        print(
            f"  control_relationship: {relationship.controlled_quantity} "
            f"<- {relationship.control_input.value}"
        )

print("\nAdvisory Reference (比較専用):")
for advisory in maneuver_package.advisory_references:
    print([(item.quantity, item.value, item.unit) for item in advisory.values])

### 2. 初期状態・風・明示的な仮定を入力する

まず変更するセルです。値を変えたら、このセル以降を順番に再実行します。`interpret_unspecified_airspeed_as_tas` などは原資料の値ではなく、現在の model gap を埋める明示的な仮定です。

In [ ]:
# InitialState: 編集可能
initial_latitude_deg = 34.7500
initial_longitude_deg = 135.4500
initial_altitude_ft = 4_000.0
initial_heading_deg_true = 90.0
initial_true_airspeed_kt = 110.0
empty_aircraft_mass_kg = 1_050.0
payload_mass_kg = 180.0
initial_fuel_mass_kg = 90.0

# Environment: 気象風向は FROM、0 kt なら NoWind
wind_from_deg_true = 270.0
wind_speed_kt = 10.0
terrain_elevation_ft_msl = 0.0

# Explicit assumptions: 原資料または POH が定義していない現在の仮定
interpret_unspecified_airspeed_as_tas = True
reference_path_descent_ft = 700.0
established_power_pct = 15.0
trim_pitch_deg = -1.0
entry_duration_s = 8.0
speed_error_to_pitch_gain_deg_per_mps = 0.12
heading_error_to_bank_gain = 0.7
radial_error_gain_per_m = 0.002
maximum_intercept_angle_deg = 25.0

# Numerical / path settings
path_point_count = 361
time_step_s = 0.2
maximum_steps = 2_000

### 3. InitialState を組み立てる

重量は Environment ではなく、機体・搭載物・残燃料から計算します。

In [ ]:
initial_state = InitialState(
    time_s=0.0,
    position=GeoPosition(initial_latitude_deg, initial_longitude_deg),
    altitude_m=feet_to_metres(initial_altitude_ft),
    heading_true_rad=degrees_to_radians(initial_heading_deg_true),
    true_airspeed_mps=knots_to_metres_per_second(initial_true_airspeed_kt),
    loading=Loading(
        empty_aircraft_mass_kg=empty_aircraft_mass_kg,
        payload=(MassItem("notebook payload", payload_mass_kg),),
    ),
    initial_fuel_mass_kg=initial_fuel_mass_kg,
)
print(f"Initial weight: {initial_state.initial_weight_kg:.1f} kg")

### 4. Environment を組み立てる

Reference Path には風を渡しません。風は Environment にだけ保持し、Trajectory の計算時に適用します。

In [ ]:
wind = (
    NoWind()
    if wind_speed_kt == 0.0
    else ConstantWind.from_meteorological_knots(
        from_direction_deg_true=wind_from_deg_true,
        speed_kt=wind_speed_kt,
    )
)
environment = Environment(
    atmosphere=Atmosphere(
        temperature_k=288.15,
        pressure_altitude_m=initial_state.altitude_m,
    ),
    wind=wind,
    terrain=FlatTerrain(feet_to_metres(terrain_elevation_ft_msl)),
)
print(type(environment.wind).__name__)

### 5. 風と独立した Reference Path を作る

110 kt の airspeed kind は確認済み本文だけでは特定できません。このデモでは上の入力セルで TAS と解釈する仮定を明示しています。旋回半径は本文の nominal Bank と解析式から求めます。

In [ ]:
if not interpret_unspecified_airspeed_as_tas:
    raise ValueError("このデモには、未特定の 110 kt を TAS と解釈する明示的な仮定が必要です")

entry_target = next(
    item
    for item in maneuver_spec.phase("entry").targets
    if item.quantity == "airspeed_at_pylon_abeam"
)
nominal_bank = next(
    item
    for item in maneuver_spec.phase("execution").nominals
    if item.quantity == "bank"
)
completion = next(
    item
    for item in maneuver_spec.termination_conditions
    if item.quantity == "accumulated_turn"
)
procedure_turn_rad = degrees_to_radians(completion.value)
target_tas_mps = knots_to_metres_per_second(entry_target.value)
reference_radius_m = abs(
    coordinated_turn_radius_m(
        target_tas_mps,
        degrees_to_radians(nominal_bank.value),
    )
)
path_center = displace_position(
    initial_state.position,
    east_m=0.0,
    north_m=-reference_radius_m,
)
reference_path = PylonSpiralPath(
    name="Notebook two-turn pylon Reference Path",
    center=path_center,
    radius_m=reference_radius_m,
    start_bearing_rad=0.0,
    sweep_rad=procedure_turn_rad,
    start_altitude_m=initial_state.altitude_m,
    end_altitude_m=initial_state.altitude_m - feet_to_metres(reference_path_descent_ft),
    point_count=path_point_count,
)
print(f"Reference radius: {reference_radius_m:.1f} m")

### 6. 仮定モデルと Guidance を組み立てる

`build_assumption_model()` は POH 降下性能表の代替ではありません。Guidance は課目本文の Target / Limit / Control Relationship を読み、下の gain と trim は明示的な仮定として受け取ります。

In [ ]:
aircraft_model = build_assumption_model()
guidance_config = SpiralGuidanceConfig(
    interpret_unspecified_airspeed_as_tas=interpret_unspecified_airspeed_as_tas,
    entry_duration_s=entry_duration_s,
    established_power_fraction=established_power_pct / 100.0,
    trim_pitch_rad=degrees_to_radians(trim_pitch_deg),
    speed_error_to_pitch_gain_rad_per_mps=degrees_to_radians(
        speed_error_to_pitch_gain_deg_per_mps
    ),
    heading_error_to_bank_gain=heading_error_to_bank_gain,
    radial_error_gain_per_m=radial_error_gain_per_m,
    maximum_intercept_angle_rad=degrees_to_radians(maximum_intercept_angle_deg),
    flap=FlapSetting.RETRACTED,
)
print(aircraft_model.name)

### 7. Guided simulation を実行する

In [ ]:
guided_result = simulate_guided_spiral_descent(
    initial=initial_state,
    environment=environment,
    maneuver_spec=maneuver_spec,
    reference_path=reference_path,
    guidance_config=guidance_config,
    aircraft_model=aircraft_model,
    termination=AccumulatedTurn(procedure_turn_rad),
    simulation_config=SimulationConfig(dt_s=time_step_s, max_steps=maximum_steps),
)
simulation = guided_result.simulation
trajectory = simulation.trajectory
print(f"outcome={simulation.outcome.value}, samples={len(trajectory)}")

## Checks

### 8. 終了理由・高度・速度・燃料・根拠ラベルを確認する

In [ ]:
termination_event = simulation.termination_event
summary = {
    "outcome": simulation.outcome.value,
    "termination_condition": termination_event.condition if termination_event else None,
    "samples": len(trajectory),
    "elapsed_s": trajectory.final.time_s - trajectory.initial.time_s,
    "initial_altitude_ft": metres_to_feet(trajectory.initial.altitude_m),
    "final_altitude_ft": metres_to_feet(trajectory.final.altitude_m),
    "final_tas_kt": metres_per_second_to_knots(trajectory.final.true_airspeed_mps),
    "final_vertical_speed_fpm": metres_per_second_to_feet_per_minute(
        trajectory.final.vertical_speed_mps
    ),
    "fuel_burned_kg": trajectory.final.fuel_burned_kg,
    "evidence": [item.value for item in trajectory.evidence],
}
summary

In [ ]:
assert EvidenceKind.ASSUMED in trajectory.evidence
assert EvidenceKind.PHYSICS_DERIVED in trajectory.evidence
assert trajectory.final.time_s > trajectory.initial.time_s
assert trajectory.final.fuel_remaining_kg < trajectory.initial.fuel_remaining_kg
print("基本チェック: OK（assumed / physics_derived を保持）")

### 9. Reference Path と Trajectory を重ねて確認する

破線が風に依存しない Reference Path、実線が Environment の風を受けた Trajectory です。

In [ ]:
ground_track_figure, _ = plot_ground_track(
    trajectory,
    reference_path=reference_path,
)
ground_track_path = artifact_dir / "guided-ground-track.png"
ground_track_figure.savefig(ground_track_path, dpi=160, bbox_inches="tight")
display(ground_track_figure)
plt.close(ground_track_figure)
print(f"saved: {ground_track_path}")

In [ ]:
altitude_figure, _ = plot_altitude_time(trajectory)
altitude_path = artifact_dir / "guided-altitude-time.png"
altitude_figure.savefig(altitude_path, dpi=160, bbox_inches="tight")
display(altitude_figure)
plt.close(altitude_figure)
print(f"saved: {altitude_path}")

In [ ]:
trajectory_3d_figure, _ = plot_trajectory_3d(
    trajectory,
    reference_path=reference_path,
)
trajectory_3d_path = artifact_dir / "guided-trajectory-3d.png"
trajectory_3d_figure.savefig(trajectory_3d_path, dpi=160, bbox_inches="tight")
display(trajectory_3d_figure)
plt.close(trajectory_3d_figure)
print(f"saved: {trajectory_3d_path}")

### 10. CSV / KML を保存する

CSV は各時刻の状態量、KML は Google Earth 等で表示できる 3D LineString です。Reference Path と Trajectory は別ファイルに保存します。

In [ ]:
csv_path = write_trajectory_csv(
    trajectory,
    artifact_dir / "guided-trajectory.csv",
)
trajectory_kml_path = write_kml(
    trajectory_to_kml(trajectory, name="Notebook Guided Trajectory"),
    artifact_dir / "guided-trajectory.kml",
)
reference_kml_path = write_kml(
    reference_path_to_kml(reference_path, name="Notebook Reference Path"),
    artifact_dir / "guided-reference-path.kml",
)
print("saved:")
for path in (csv_path, trajectory_kml_path, reference_kml_path):
    print(f"  {path}")

## Next Steps

- `wind_speed_kt = 0.0` と 10.0 で実行し、同じ Reference Path に対する Trajectory の差を比較する。
- 初期高度や風向を変え、終了理由が `accumulated_turn` か `minimum_agl` か確認する。
- Guidance ではなく固定 `Pitch / Bank / PWR / Flap` を試す場合は `simulate_forward(...)` を使う。固定入力実験を公式課目の再現とは表示しない。
- 実行後は `File > Save Notebook` でセル出力を Notebook に保存する。CSV / KML / PNG はホスト側の `artifacts/` に残る。